In [1]:
from mspasspy.db.database import Database
import mspasspy.client as msc
mspass_client = msc.Client()
db = mspass_client.get_database("Earthscope2025")

# 1. Convert the following MsPASS code fragment to do the same operation in parallel with dask (the scheduler we used in the notebook exercise) using the bag.from_sequence method we used in the first example on geolab.

In [ ]:
#Code to convert
cursor = db.wf_Seismogram.find(query)
for doc in cursor:    
    d = db.read_data(doc,collection="wf_Seismogram",normalize=[site_matcher,source_matcher])
    d = detrend(d,type="constant")
    d = filter(d,type="bandpass",freqmin=0.02,freqmax=1.5)
    d = rotate_to_standard(d)
    d = myfunction(d,x=2.0,y=42)
    db.save_data(d)

In [ ]:
#Converted Code
cursor = db.wf_Seismogram.find(query)
dataset = dbg.from_sequence(cursor,npartitions=50)
dataset = dataset.map(db.read_data,collection='wf_Seismogram', normalize=[site_matcher,source_matcher])
dataset = dataset.map(detrend, type="constant")
dataset = dataset.map(filter, type="bandpass",freqmin=0.02,freqmax=1.5)
dataset = dataset.map(rotate_to_standard)
dataset = dataset.map(myfunction,x=2.0,y=42)
dataset = dataset.map(set_fname_attributes,dir=dir)
dataset = dataset.map(db.save_data,collection='wf_Seismogram',storage_mode='file',data_tag='parallel_output')
dataset=dataset.compute()

In [ ]:
#Serial 
cursor=db.wf_Seismogram.find(query).limit(N)
t0 = time.time()
for doc in cursor:
    d =db.read_data(doc,collection='wf_Seismogram')
    d = rotate_to_standard(d)
    d = apply_FST(d)
    d = broadband_snr_QC(d,
                      component=2,
                      use_measured_arrival_time=True,
                      measured_arrival_time_key="Ptime",
                      noise_window=noise_window,
                      kill_null_signals=True,
                     )
t=time.time()
print("serial time to process ",N," Seismogram objects=",t-t0)

#Parallel
cursor = db.wf_Seismogram.find(query)
dataset = dbg.from_sequence(cursor,npartitions=50)
dataset = dataset.map(db.read_data,collection='wf_Seismogram')
dataset = dataset.map(rotate_to_standard)
dataset = dataset.map(apply_FST)
dataset = dataset.map(broadband_snr_QC,
                    component=2,
                    use_measured_arrival_time=True,
                    measured_arrival_time_key="Ptime",
                    noise_window=noise_window,
                    kill_null_signals=True,
                    )
dataset = dataset.map(set_fname_attributes,dir=dir)
dataset = dataset.map(db.save_data,collection='wf_Seismogram',storage_mode='file',data_tag='parallel_output')
dataset=dataset.compute()

# 2. Rewrite your answer to the previous question using read_distributed_data as described in geolab notebook

In [ ]:
#Code using read_distributed_data
n=db.wf_Seismogram.count_documents(query)
print("This is to demonstrate the start of the conversion of code and the job will process ",n," Seismogram objects")
dataset = read_distributed_data(db,query=query,collection='wf_Seismogram',normalize=[site_matcher,source_matcher], npartitions=50)
dataset = dataset.map(detrend, type="constant")
dataset = dataset.map(filter, type="bandpass",freqmin=0.02,freqmax=1.5)
dataset = dataset.map(rotate_to_standard)
dataset = dataset.map(myfunction,x=2.0,y=42)
dataset = dataset.map(set_fname_attributes,dir=dir)
ret = write_distributed_data(dataset,db,collection='wf_Seismogram',storage_mode='file',data_tag='parallel_output_variant1')

# 3. Parallelize this code segment to process a large ensemble, ensemble, in parallel.   Note you can find examples of this approach in the notebook we ran on frontera. 

In [ ]:
#Code to convert
for sid in sidlist:
    query={"source_id" : sid}
    cursor = db.wf_TimeSeries.find(query)
    ensemble = db.read_data(cursor,normalize=[chan_matcher,source_matcher])
    
    for i in range(len(ensemble.member)):
        ensemble.member[i] = demean(ensemble.member[i],type="constant")
        ensemble.member[i] = filter(ensemble.member[i],type="lowpass",freq=2.0)
    db.save_data(ensemble)

In [ ]:
#Converted code to process a large esemble in parallel
for sid in srcidlist:
    print("working on  ",sid)
    query = {"source_id" : sid}
    mydata = read_distributed_data(db,
                                   query=query,
                                   collection="wf_TimeSeries",
                                   normalize=[chan_matcher,source_matcher],
                                   npartitions=60,
                                  )
    mydata=mydata.map(demean, type = "constant")
    mydata=mydata.map(filter, type = "lowpass",freq=2.0)
    dlist = mydata.compute()
    
    # package into ensemble for a faster write 
    ens=TimeSeriesEnsemble(len(dlist))
    for d in dlist:
        ens.member.append(d)
    # needed because we constructed this manual from a list
    ens.set_live()
    ens = set_file_path(ens,dir=tsdir)
    ens = db.save_data(ens,
                       return_data=True,
                       collection="wf_TimeSeries",
                       storage_mode="file",
                       dir=tsdir,
                       data_tag="preprocessed_map",
                       )
    del ens